# Week 1 - Exercises

## Exercise 1: Design your own Conputational Social Science project

question: how do major privacy events (data breaches, policy changes) affect discussion volume and topics in online privacy communities?

data source: reddit api, posts and comments from r/privacy, r/degoogle, etc.

unit of analysis: posts/comments

measurable variable: post volume per week, looking for spikes after major events

ethical concern: even though reddit posts are technically public, users often share sensitive personal info (political views, security setup, location details) without expecting it to be used for research. this ties into nissenbaum's idea of context-relative informational norms from salganik ch. 6, just because data is public doesn't mean people are okay with it being studied.

## Exercise 2: Understanding Conputational Social Science in a data-driven way

data source: publication and citation data from OpenAlex or Semantic Scholar API, both free, no auth needed, good coverage

what it helps us understand: who the most productive/impactful researchers are, which institutions are most active, and how the field's output has grown over time

how to collect it: query the api for publications by researchers who appear at CSS conferences (like the ic2s2 list from exercise 3, it all connects)

limitation/bias: publication count alone doesn't mean much, someone with 50 papers might be less impactful than someone with 5 highly cited ones. also favors senior researchers who've just had more time. including citation count (and maybe something like average citations per paper) helps balance this out, but even citations have their own biases

## Exercise 3: Web-scraping IC2S2 2025

In [1]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup

### Scraping

In [2]:
# start with organizers page since its static html, all names should be directly in the page source.
org_url = "https://ic2s2-2025.org/organization/"
org_response = requests.get(org_url)
org_soup = BeautifulSoup(org_response.text, "html.parser")

# by inspecting the page in a browser we can see all names are in h3 tags
for tag in org_soup.find_all("h3"):
    print(repr(tag.get_text(strip=True)))

'Marc Keuschnigg'
'Sonia Yeh'
'Peter Hedström'
'Nina Tahmasebi'
'Elliot Ash'
'Hendrik Erz'
'Mirta Galesic'
'David Garcia'
'Josef Ginnerskov'
'Jacob Habinek'
'Petter Holme'
'Andreas Jungherr'
'Juhi Kulshrestha'
'Erika Fille T. Legara'
'May Lim'
'Laura Lotero'
'Angélica Sousa da Mata'
'Kwabena Afriyie Owusu'
'Denise Pumain'
'Yang Yue'
'Martin Arvidsson'
'Yuan Liao'
'Hendrik Erz'
'Yuan Liao'
'Anastasia Menshikova'
'José Bener'
'Johannes Fagerlind'
'Lauren Fahey'
'Zhao Feng'
'Thomas Haase'
'Yukun Jiao'
'Xiaojing Jie'
'Kazuki Sakamoto'
'Helpis Saleh'
'Marc Sparhuber'
'Vsevolod Suschevskiy'
'Yun-Tsz Tsai (Clara)'
'Yize Zhang'
'Yue Zheng'
'Noé Vidal-Naquet'
'Address'
'Social'
'Email'


In [3]:
org_names = []
for h3 in org_soup.find_all("h3")[:-3]:  # last 3 are footer: Address, Social, Email
    name = h3.get_text(strip=True)
    if name:
        org_names.append(name)

print(f"Names from organization page: {len(org_names)}")
for name in sorted(org_names):
    print(f"  {name}")

Names from organization page: 40
  Anastasia Menshikova
  Andreas Jungherr
  Angélica Sousa da Mata
  David Garcia
  Denise Pumain
  Elliot Ash
  Erika Fille T. Legara
  Helpis Saleh
  Hendrik Erz
  Hendrik Erz
  Jacob Habinek
  Johannes Fagerlind
  Josef Ginnerskov
  José Bener
  Juhi Kulshrestha
  Kazuki Sakamoto
  Kwabena Afriyie Owusu
  Laura Lotero
  Lauren Fahey
  Marc Keuschnigg
  Marc Sparhuber
  Martin Arvidsson
  May Lim
  Mirta Galesic
  Nina Tahmasebi
  Noé Vidal-Naquet
  Peter Hedström
  Petter Holme
  Sonia Yeh
  Thomas Haase
  Vsevolod Suschevskiy
  Xiaojing Jie
  Yang Yue
  Yize Zhang
  Yuan Liao
  Yuan Liao
  Yue Zheng
  Yukun Jiao
  Yun-Tsz Tsai (Clara)
  Zhao Feng


In [4]:
# by inspecting the program page we can see that there is a summary table with keynote speaker name

prog_url = "https://ic2s2-2025.org/program/"
prog_response = requests.get(prog_url, headers={"User-Agent": "Mozilla/5.0"})
prog_soup = BeautifulSoup(prog_response.text, "html.parser")

# The overview table has keynote info in cells like "Keynote: Name"
for td in prog_soup.find_all("td"):
    text = td.get_text(strip=True)
    if "keynote" in text.lower():
        print(repr(text))

'Keynote: Dean EcklesKeynote: Kathleen Carley'
'Keynote: Laura NelsonLightning talks'
'Keynote: Duncan J. WattsLightning talks'
'Keynote: Brandon StewartKeynote: Lisa P. Argyle'
'Keynote: Amir GoldbergKeynote: Arnout van de Rijt'
'Keynote: Sarah WilliamsClosing remarks'


In [5]:
keynote_names = []
for td in prog_soup.find_all("td"):
    text = td.get_text()
    if "Keynote:" in text:
        # Each cell can have multiple "Keynote: Name" entries
        for segment in text.split("Keynote:")[1:]:
            name = segment.strip().split("\n")[0].strip()
            if name:
                keynote_names.append(name)

print(f"Keynote speakers found: {len(keynote_names)}")
for name in keynote_names:
    print(f"  {name}")

Keynote speakers found: 9
  Dean Eckles
  Kathleen Carley
  Laura Nelson
  Duncan J. Watts
  Brandon Stewart
  Lisa P. Argyle
  Amir Goldberg
  Arnout van de Rijt
  Sarah Williams


### Extract csv

In [6]:
# the program page has interactive parts that is loaded from a csv file which we found by using browser dev tools, network tab

from io import StringIO

csv_url = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"
csv_response = requests.get(csv_url, headers={"User-Agent": "Mozilla/5.0"})
schedule_df = pd.read_csv(StringIO(csv_response.text))

print("Columns:", schedule_df.columns.tolist())
print(f"Rows: {len(schedule_df)}")
schedule_df[["type", "author", "chair"]].head(10)

Columns: ['date_start', 'date_end', 'type', 'title', 'abstract', 'author', 'location', 'session', 'session_order', 'chair', 'openreview_id', 'notes']
Rows: 721


,type,author,chair
0,single,"Étienne Ollion, Émilien Schultz",NaN
1,single,"Miriam Schirmer, Julia Mendelsohn, Dustin Wrig...",NaN
2,single,Caspar van Lissa,NaN
3,single,"Jorge Barreras, Thomas Li, Chen Zhong, Cate He...",NaN
4,single,"Paolo Turrini, Elias Fernández Domingos",NaN
5,NaN,NaN,NaN
6,single,"Kristina Gligoric, Cinoo Lee, Tijana Zrnic",NaN
7,single,"Adel Daoud, Connor Jerzak",NaN
8,single,"Mark Whiting, Linnea Gandhi, Amirhossein Nakha...",NaN
9,single,"Egor Kotov, Johannes Mast",NaN


In [7]:
def extract_names(cell):
    """Split a cell that may contain multiple comma/semicolon-separated names."""
    if pd.isna(cell):
        return []
    text = str(cell)
    # Normalize common separators
    text = text.replace(" & ", ", ").replace(" and ", ", ")
    parts = re.split(r"[;,|]", text)
    cleaned = []
    for p in parts:
        name = " ".join(p.split()).strip()  # collapse whitespace
        if name:
            cleaned.append(name)
    return cleaned

csv_names = set()
for col in ["author", "chair"]:
    for cell in schedule_df[col]:
        for name in extract_names(cell):
            csv_names.add(name)

print(f"Names from CSV (author + chair): {len(csv_names)}")

Names from CSV (author + chair): 1574


### Merge and dedup

In [8]:
# now we merge all names and deduplicates

def normalize(name):
    """Lowercase, strip dots and extra whitespace for comparison."""
    n = name.replace(".", "")
    n = re.sub(r"\s+", " ", n).strip()
    return n.lower()

# combine names from all sources
all_raw = list(csv_names) + org_names + keynote_names
print(f"Total raw names (all sources): {len(all_raw)}")


# exact dedup, first occurrence wins, duplicates gets logged
seen = {}
removed = []
for name in all_raw:
    key = normalize(name)
    if key in seen:
        removed.append((name, seen[key])) # track what got dropped and what it matched
    else:
        seen[key] = name # keep the first version encountered

# sorted list of unique names
deduped = sorted(seen.values())
print(f"After exact dedup: {len(deduped)}")

# print all removed names for manual review
print(f"\nRemoved {len(removed)} duplicates:")
for dropped, kept in sorted(removed, key=lambda x: x[1].lower()):
    print(f"  kept '{kept}'  |  dropped '{dropped}'")

Total raw names (all sources): 1623
After exact dedup: 1597

Removed 26 duplicates:
  kept 'Amir Goldberg'  |  dropped 'Amir Goldberg'
  kept 'Anastasia Menshikova'  |  dropped 'Anastasia Menshikova'
  kept 'Arnout van de Rijt'  |  dropped 'Arnout van de Rijt'
  kept 'Brandon Stewart'  |  dropped 'Brandon Stewart'
  kept 'David Garcia'  |  dropped 'David Garcia'
  kept 'Dean Eckles'  |  dropped 'Dean Eckles'
  kept 'Duncan J. Watts'  |  dropped 'Duncan J. Watts'
  kept 'Elena D'Agnese'  |  dropped 'Elena D'agnese'
  kept 'Hendrik Erz'  |  dropped 'Hendrik Erz'
  kept 'Hendrik Erz'  |  dropped 'Hendrik Erz'
  kept 'Jacob Habinek'  |  dropped 'Jacob Habinek'
  kept 'Jia Li'  |  dropped 'Jia LI'
  kept 'Josef Ginnerskov'  |  dropped 'Josef Ginnerskov'
  kept 'Juhi Kulshrestha'  |  dropped 'Juhi Kulshrestha'
  kept 'Kazuki Sakamoto'  |  dropped 'Kazuki Sakamoto'
  kept 'Laura Nelson'  |  dropped 'Laura Nelson'
  kept 'Lisa P. Argyle'  |  dropped 'Lisa P. Argyle'
  kept 'Marc Keuschnigg'  |

In [9]:
# some names may appear with slightly different spellings
# using rapidfuzz to find near-duplicates
# and again print them first so we can review before removing anything

import unicodedata
from rapidfuzz import fuzz

def norm_fuzzy(name):
    """Aggressive normalization for fuzzy comparison."""
    n = unicodedata.normalize("NFKD", name)
    n = "".join(c for c in n if not unicodedata.combining(c))  # strip accents
    n = n.replace(".", "").replace("'", "").replace("'", "")
    n = re.sub(r"\s+", " ", n).strip().lower()
    return n

# Compare consecutive names (sorted) — catches most near-dupes efficiently
# experimented with different threshold, i found 70 to be a good balance of not too much manual review
# and still catching most if not all of the near duplicates
print("Potential duplicates (score > 70):\n")
for a, b in zip(deduped, deduped[1:]):
    score = fuzz.token_sort_ratio(norm_fuzzy(a), norm_fuzzy(b))
    if score > 70:
        print(f"  {score}:  '{a}'  <->  '{b}'")

Potential duplicates (score > 70):

  75.67567567567568:  'Agnieszka Czaplicka'  <->  'Agnieszka Falenska'
  75.0:  'Akiko Matsuo'  <->  'Akira Matsui'
  73.6842105263158:  'Alessandro Flammini'  <->  'Alessandro Galeazzi'
  76.47058823529412:  'Alessandro Galeazzi'  <->  'Alessandro Lomi'
  76.47058823529412:  'Alessandro Lomi'  <->  'Alessandro Pluchino'
  72.22222222222221:  'Alessandro Pluchino'  <->  'Alessandro Riboni'
  72.22222222222221:  'Anastasia Giachanou'  <->  'Anastasia Golovin'
  73.33333333333334:  'Andrew Halterman'  <->  'Andrew Lippman'
  71.42857142857143:  'Andrew Piper'  <->  'Andrew Renninger'
  72.72727272727273:  'Anna Beers'  <->  'Anna Bertani'
  71.42857142857143:  'Babak Hemmatian'  <->  'Babak Heydari'
  93.75:  'Brandon M. Stewart'  <->  'Brandon Stewart'
  72.0:  'Byunghwee Lee'  <->  'Byungkyu Lee'
  72.0:  'Cameron Hickey'  <->  'Cameron Lai'
  94.11764705882352:  'Caspar J. Van Lissa'  <->  'Caspar van Lissa'
  75.0:  'Christian Geiß'  <->  'Christia

In [10]:
# true duplicates identified by reviewing Cell 9 output
# keep the more complete spelled variant.
to_remove = [
    "Brandon Stewart",            # keep "Brandon M. Stewart"
    "Caspar van Lissa",           # keep "Caspar J. Van Lissa"
    "Connor Jerzak",              # keep "Connor Thomas Jerzak"
    "Cristian Candia",            # keep "Cristian E Candia"
    "Daniel O\u2019Brien",        # curly quote — keep "Daniel O'Brien"
    "Douglas Guilbeault",         # keep "Douglas Richard Guilbeault"
    "Duncan Watts",               # keep "Duncan J. Watts"
    "Kathleen Carly",             # typo — keep "Kathleen M. Carley"
    "Kathleen Carley",            # shorter — keep "Kathleen M. Carley"
    "Kristina Gligoric",          # keep "Kristina Gligorić" (with accent)
    "Przemyslaw Grabowicz",       # keep "Przemyslaw A. Grabowicz"
    "Sandro Sousa",               # keep "Sandro Ferreira Sousa"
    "Sungkyu Park",               # keep "Sungkyu Shaun Park"
]

final_names = sorted([n for n in deduped if n not in to_remove])
print(f"Removed {len(deduped) - len(final_names)} fuzzy duplicates")
print(f"Final unique researchers: {len(final_names)}")

Removed 13 fuzzy duplicates
Final unique researchers: 1584


In [11]:
pd.Series(final_names, name="name").to_csv("week1_ic2s2_2025_researchers.csv", index=False)
print(f"Saved {len(final_names)} names to week1_ic2s2_2025_researchers.csv")

Saved 1584 names to week1_ic2s2_2025_researchers.csv


### Process explanation

We scraped and extracted names from three places:

1. Organization page, we scraped the HTML and extracted names from `<h3>` tags using BeautifulSoup, the last three items were footer items so we just sliced those off.

2. Program overview table, the static HTML has a overview table where keynote speakers appear as "Keynote: Name", so  we split those to extract the speaker names.

3. Schedule CSV, the interactive agenda on the program page is powered by Conferia.js, which loads all its data from a CSV file. We found it by inspecting the page source. The `author` and `chair` columns had all the talk/poster/session names, separated by commas, semicolons, or pipes.

For deduplication we did two passes:

First an exact normalization, lowercase everything, strip dots, collapse whitespace. This caught the obvious duplicates like the same person showing up in both the CSV and the org page. We printed every removal to make sure nothing weird got merged.

Then a fuzzy pass using `rapidfuzz` with a threshold of 70. This flagged a bunch of pairs, but most were false positives, just different people with similar names (lots of Alessandros, Michaels, etc.). We went through the list manually and only removed the ones that I deemed the same person.

It's worth noting that even cases that look obvious, like "Kathleen Carly" vs "Kathleen M. Carley" — could in theory be two different people. There's a fine line between cleaning up genuine duplicates and accidentally merging distinct researchers, so some judgment calls are unavoidable.